In [ ]:
import os
import pathlib
import json

import pandas as pd

In [ ]:
BASE_DIR = str(pathlib.Path().resolve())
TARGET_DIR = ''
WORKING_DIR = f'{BASE_DIR}/{TARGET_DIR}'

dir_list = os.listdir(WORKING_DIR)

mapping = ''
mapping_dict = {}

i = 1

filtered_files = [
    filename for filename in dir_list
        if filename.endswith('.xlsx')
    ]

mapping_dict = {
    number + 1: filename for number, filename
        in enumerate(filtered_files)
    }

#
print(f'BASE_DIR: {BASE_DIR}')
print(f'TARGET_DIR: {TARGET_DIR}')
print(f'WORKING_DIR: {WORKING_DIR}')
print(f'dir_list: {dir_list}')

In [ ]:
if not mapping_dict:
        print('\n==================================================')
        print(f'{"!В папке отсутствует mapping файл формата xlsx!":^50}')
        print('==================================================')
        
        exit()
elif len(mapping_dict) == 1:
        mapping_filename = next(iter(mapping_dict.values()))
elif len(mapping_dict) > 1:
        for key in mapping_dict.keys():
                print(key,': ', mapping_dict[key])
        
        mapping_key = input('\nWrite mapping number: ')

        mapping_filename = mapping_dict[int(mapping_key)]

try:
        print("Reading mapping file...")
        main_df = pd.read_excel(f'{WORKING_DIR}/{mapping_filename}',
                                sheet_name='Mapping',
                                usecols="D,"  # Source Schema
                                        "E,"  # Source Table
                                        "G,"  # Source Code
                                        "I,"  # Source Data Type
                                        "J,"  # Source Length
                                        "T,"  # Target Schema
                                        "U,"  # Tagret Table
                                        "V,"  # Target Code
                                        "Z,"  # Target Data Type
                                        "AA") # Target Length
except Exception as e:
        print("Error while reading file: ", e)
        

main_df = main_df.drop(0,axis=0)

main_df.columns = ['SchemaS', 'TableS', 'CodeS',
                        'DataTypeS', 'LengthS',
                        'SchemaT', 'TableT', 'CodeT',
                        'DataTypeT', 'LengthT']

main_df = main_df.fillna('')

main_df.head()

In [ ]:
main_df['SchemaS'] = main_df['SchemaS'].apply(lambda x: str(x).strip())
main_df['TableS'] = main_df['TableS'].apply(lambda x: str(x).strip())
main_df['CodeS'] = main_df['CodeS'].apply(lambda x: str(x).strip())
main_df['DataTypeS'] = main_df['DataTypeS']\
                                .apply(lambda x: str(x).strip())
main_df['LengthS'] = main_df['LengthS'].apply(lambda x: str(x).strip())
#
main_df['SchemaT'] = main_df['SchemaT'].apply(lambda x: str(x).strip())
main_df['TableT'] = main_df['TableT'].apply(lambda x: str(x).strip())
main_df['CodeT'] = main_df['CodeT'].apply(lambda x: str(x).strip())
main_df['DataTypeT'] = main_df['DataTypeT']\
                                .apply(lambda x: str(x).strip())
main_df['LengthT'] = main_df['LengthT'].apply(lambda x: str(x).strip())

#
main_df = main_df[~main_df['CodeT'].isin(['hdp_processed_dttm'])]
main_df = main_df[main_df['CodeT']!='']

In [ ]:
IsCBlobTableIgnore = 1
TakeOnlyCBlobTables = 2
IsCBlobColumnIgnore = 1

In [ ]:
if IsCBlobTableIgnore == 1:
    cblob_table_df = main_df[
            main_df['DataTypeS'].isin(['CLOB', 'BLOB'])
        ]
    ignore_cblob_table_list = cblob_table_df['TableS'].unique().tolist()
    if ignore_cblob_table_list:
        main_df = main_df[~main_df['TableS']\
                            .isin(ignore_cblob_table_list)]
elif TakeOnlyCBlobTables == 1:
    cblob_table_df = main_df[
            main_df['DataTypeS'].isin(['CLOB', 'BLOB'])
        ]
    cblob_table_list = cblob_table_df['TableS'].unique().tolist()
    if cblob_table_list:
        main_df = main_df[main_df['TableS']\
                            .isin(cblob_table_list)]
    if IsCBlobColumnIgnore == 1:
        main_df = main_df[~main_df['DataTypeS'].isin(['CLOB', 'BLOB'])]
elif IsCBlobColumnIgnore == 1:
    main_df = main_df[~main_df['DataTypeS'].isin(['CLOB', 'BLOB'])]

In [ ]:
custom_schema_s_name = ''
custom_schema_t_name = ''
table_type_filter = 'TableS'
code_type_filter = 'CodeS'
take_only_table_list = []
ignore_table_list = []
ignore_code_list = []

In [ ]:
if custom_schema_s_name:
    main_df['SchemaS'] = custom_schema_s_name
# elif env_type == 1:
#     main_df['SchemaS'] = local_vars_dict['custom_schema_s_name']

if custom_schema_t_name:
    main_df['SchemaT'] = custom_schema_t_name
# elif env_type == 1:
#     main_df['SchemaT'] = local_vars_dict['custom_schema_t_name']

if take_only_table_list:
    main_df = main_df[main_df[table_type_filter].isin(take_only_table_list)]

if ignore_table_list:
    main_df = main_df[~main_df[table_type_filter].isin(ignore_table_list)]

if ignore_code_list:
    main_df = main_df[~main_df[code_type_filter].isin(ignore_code_list)]

In [ ]:
main_df['schemaS.tableS'] = main_df['SchemaS'] +'.'+ main_df['TableS']

main_df = main_df.sort_values(['TableS'])

main_df.index = range(1, len(main_df) + 1)

main_df = main_df

print(main_df.head())

In [ ]:
schema_t = main_df.iloc[0]['SchemaT']
schema_t

In [ ]:
current_df = main_df[
    main_df['schemaS.tableS'] == "NORMDOC.A3_ACCESS_KEYS"
    ]

print(current_df.head())

schema_s = current_df.iloc[0]['SchemaS']
print(schema_s)
source_table = current_df.iloc[0]['TableS']
print(source_table)
target_table = current_df.iloc[0]['TableT']
print(target_table)

In [ ]:
def print_results(self, schema_t, test_flow_entity_lst, schtbl_num):
    # print result to file
    print('=PRINT RESULT=')

    main_json_template = {
        "connection": {
            "connType": "jdbc",
            "url": "jdbc:oracle:thin:@192.168.1.67:1521:FREE",
            "driver": "oracle.jdbc.driver.OracleDriver",
            "user": "ibank2",
            "password": "ibank2"
        },
        "commonInfo": {
            "targetSchema": schema_t,
            "etlSchema": schema_t,
            "logsTable": "logs556"
        },
        "flows": test_flow_entity_lst
        }

    """
    main_json_template_default = {
        "connection": {
            "connType": "jdbc",
            "url": "...",
            "driver": "...",
            "user": "...",
            "password": "..."
        },
        "commonInfo": {
            "targetSchema": schema_t,
            "etlSchema": schema_t,
            "logsTable": "logs..."
        },
        "flows": test_flow_entity_lst
        }
    """
    
    prefix = """spark-submit --master yarn --conf spark.master=yarn --conf spark.submit.deployMode=cluster --conf spark.yarn.maxAppAttempts=1 --conf spark.sql.broadcastTimeout=600 --conf spark.hadoop.hive.exec.dynamic.partition=true --conf spark.hadoop.hive.exec.dynamic.partition.mode=nonstrict --conf spark.driver.userClassPathFirst=true --conf spark.executor.userClassPathFirst=true --jars /home/hdoop/drivers/jcc-11.5.9.0.jar,/home/hdoop/drivers/commons-pool2-2.11.0.jar,/home/hdoop/drivers/delta-core_2.13-2.2.0.jar,/home/hdoop/drivers/delta-storage-2.2.0.jar,/home/hdoop/drivers/mssql-jdbc-9.2.1.jre8.jar,/home/hdoop/drivers/ojdbc8-21.6.0.0.1.jar,/home/hdoop/drivers/orai18n-19.3.0.0.jar,/home/hdoop/drivers/org.apache.servicemix.bundles.kafka-clients-2.4.1_1.jar,/home/hdoop/drivers/postgresql-42.3.1.jar,/home/hdoop/drivers/spark-sql-kafka-0-10_2.13-3.3.2.jar,/home/hdoop/drivers/spark-token-provider-kafka-0-10_2.13-3.3.2.jar,/home/hdoop/drivers/vertica-jdbc-11.1.0-0.jar,/home/hdoop/drivers/xdb6-18.3.0.0.jar,/home/hdoop/drivers/xmlparserv2-19.3.0.0.jar --class sparketl.Main /home/hdoop/SparkEtl_ora.jar ' """

    suffix = " '"

    res_json = json.dumps(main_json_template)

    json_core = res_json.replace('}}', '} }').replace('{{', '{ {')\
                        .replace('}]', '} ]').replace('[{', '[ {')\
                        .replace(']}', '] }').replace('{[', '{ [')\
                        .replace('"}', '" }').replace('{"', '{ "')

    # define name for json
        
    results_file = str(mapping.split('.')[0])

    results_dir = (
        f'{WORKING_DIR}/{results_file}_{str(schtbl_num)}_load.sh'
        )
    
    print(f'results_dir: {results_dir}')

    #
    with open(results_dir, mode="w", encoding=enc) as write_file:
        # json.dump(main_json_template, write_file, ensure_ascii=False)
        write_file.write(prefix)
        write_file.write(json_core)
        write_file.write(suffix)
        
    print('=DONE=')

In [ ]:
schema_t = main_df.iloc[0]['SchemaT']
print(f'Target Schema: {schema_t}')
test_flow_entity_lst = []

# get schemaS.tables from mapping
# schemaS_tableS_lst = main_df['schemaS.tableS'].unique()
#
schemaS_tableS_lst = [
    "RCK.VPPOOL", "RCK.ACCEPTLIMIT", "RCK.ACCEPTLIMITACC", "RCK.ADDUSERREQUEST", "RCK.ADDUSERREQUESTBUDGETS", "RCK.ASSETHISTORY", "RCK.WEAKPASSWORD", "RCK.AUTODECLINECODE", "RCK.VPPOOLPARTY", "RCK.RPLRECORD", "RCK.CONFIRMCONTRACTOR", "RCK.TMP_NUMLIST", "RCK.TMP_STRLIST", "RCK.CARDINDEXTURN", "RCK.CCF", "RCK.CCFDOCUMENT", "RCK.CCFLINE", "RCK.CPCHARGEDPERCENTLOG", "RCK.CPCHARGESTRANSLOG", "RCK.CPMINBALANCETRANS", "RCK.CERTREVOKEDLIST", "RCK.CERTREVOKEDLISTITEM", "RCK.CERTTRANSFER", "RCK.CERTTRANSFERACCOUNT", "RCK.CLASSIFIER3", "RCK.CLASSIFIER4", "RCK.REASNOTCOORD", "RCK.REFDOC", "RCK.PAYMENTPOS", "RCK.PAYMENTPOSLINE", "RCK.PAYMENTPURPOSE", "RCK.PENDINGACTION", "RCK.SMFRBANK", "RCK.SMSDOC", "RCK.SMSREDIRECT", "RCK.ROLECLASSIFIER1", "RCK.CURRFIELD23ECODE", "RCK.DAYCOUNTCMP", "RCK.CONFIRMCONTRACTORACC", "RCK.CONTRACTGROUNDDOC", "RCK.CONTRACTGROUNDDOCTYPE", "RCK.CONTRACTPAYMENTGROUNDDOC", "RCK.DOCGROUP", "RCK.SORSNOTIFY", "RCK.SCHPLANRUNLOG", "RCK.PRIVILEGEDUSERCERT", "RCK.RATEPLACE", "RCK.ORGTYPE", "RCK.ORGUIN", "RCK.OTPLOGIN", "RCK.PANELITEM", "RCK.PAYCODE", "RCK.PAYDOCEMPLOYEE", "RCK.PAYINDEX", "RCK.OCOVERTECHPARAMS", "RCK.CPPOOL", "RCK.CPPOOLACCLINES", "RCK.CPPOOLCALCLOG", "RCK.CPPOOLSCALE", "RCK.CPPOOLSCALELINES", "RCK.CPPOOLSTATE", "RCK.CPPRCRATESCALE", "RCK.OPERATIONALCATEGORY", "RCK.OPERATIONALPLAN", "RCK.OPERATIONALPLANLINE", "RCK.ORGACCOUNTABSINFO", "RCK.ORGCONTACTPERSON", "RCK.SRGENOVERLIMNOTICE", "RCK.SROVERREQUEST", "RCK.SROVERREQUESTLINES", "RCK.SRSCHEMEIMPORTLOG", "RCK.ORGLICENCE", "RCK.DOCVISA", "RCK.STATEMENTDOCUPDATE", "RCK.STATEMENTPROCESS", "RCK.DBOMSGGROUP", "RCK.SRSTRUCTUREPARAM", "RCK.SRUNUSEDOVERLIMLOG", "RCK.MPSTRUCTURE", "RCK.MPSTRUCTUREPARAM", "RCK.MSGNOACCEPTCCF", "RCK.MOUNTEDEVENTARCHIVE", "RCK.FINANCETYPE", "RCK.FISCALNOTICE", "RCK.DOCSIGNTEMPLATE", "RCK.DOCTOCCF", "RCK.SREVENT", "RCK.SREVENTCODE", "RCK.SREXPORTLDRLOG", "RCK.SRFINRETGROUP", "RCK.SRGENLIMCALC", "RCK.SRGENLIMCALCDOCLINES", "RCK.SRGENLIMCALCRESTLINES", "RCK.SRGENLIMCALCSUPLINES", "RCK.SRGENOVERLIMLOG", "RCK.LIBORRATE", "RCK.LICTYPE", "RCK.LIMDOCDOC", "RCK.SRSTRUCTURE", "RCK.ICLBORROWER", "RCK.ICLCONTRACT", "RCK.ICLFUNDINGRULE", "RCK.ICLINTEREST", "RCK.ICLLENDER", "RCK.SWIFTBANKOPER", "RCK.SYNTANALYS", "RCK.MATPOOLRULE", "RCK.MATPOOLRULEACC", "RCK.MESSAGEFORBANK", "RCK.MFRACCOUNT", "RCK.MOBILEAPPLICATION", "RCK.ICLLOG", "RCK.ICLMEMBERCARTESIAN", "RCK.ICLREPAYMENTPERIOD", "RCK.ICLREPAYMENTRULE", "RCK.IMNS", "RCK.IMPORTFIELDS", "RCK.INTERACTINGBANK", "RCK.INTERACTINGBANKPARAM", "RCK.IPSGATE", "RCK.FRAUDMSG", "RCK.FREEDOCRECESTIMATES", "RCK.FVCHANGEALG", "RCK.FVCHANGETUNE", "RCK.GROUNDDOCTYPE", "RCK.GROUPSFORPAYCALENDAR", "RCK.LIMITSCLASS", "RCK.VPCOMPENSATIONLOG", "RCK.VPACCOUNTLOG", "RCK.USERSIGNRIGHTREQUEST", "RCK.USERPROCCONTROL", "RCK.BANKACCOUNTTYPE", "RCK.ADDUSERREQUESTROLES", "RCK.AGREEMENTROUTE", "RCK.AGREEMENTROUTEORG", "RCK.CACHEPOOLACC", "RCK.CACHEPOOLDOC", "RCK.CACHEPOOLPARAM", "RCK.AIVPCHVNS", "RCK.AIVPCHANGEREQUEST", "RCK.ROLEACCEPTLIMIT", "RCK.SORSPOOLBALANCELOG", "RCK.ROLEAMOUNT", "RCK.BALANCECORRACCOUNT", "RCK.AIVPACCPARAMDOC", "RCK.CERTREQUESTREVOKE", "RCK.BLOCKINGCODEKESR", "RCK.AIVPVNS", "RCK.AIVPCRVNS", "RCK.AIVPCREATEREQUEST", "RCK.SORSPOOLNOLOG", "RCK.RATE", "RCK.SRDISTRLDRORDERLINES", "RCK.SRDISTRLDRORDER", "RCK.SRDISTRLDRLOG", "RCK.SRCORRLIMDEBREST", "RCK.SRCONSSCHEME", "RCK.SOURCEFINANC", "RCK.ROLEERROR", "RCK.ROLEDOCSTATUS", "RCK.ROLETERRBANK", "RCK.RPL", "RCK.RPLERROR", "RCK.AGREEREJECTREASON", "RCK.REFINANCERATE", "RCK.REGISTRYNOTPAID", "RCK.REPORATE", "RCK.RESTTOABS", "RCK.RETRANSMITSITE", "RCK.SERVICEIMPORTLOG"
]
#
schtbl_len = len(schemaS_tableS_lst)
print(f"Number of source schema.tables: {schtbl_len}")

schtbl_cnt_trigger = 0
schtbl_cnt_max = 99
schtbl_num = 1

db_type = 1

if db_type == 1:  # Oracle
    # generate oracle flows
    print('=MAKING ORACLE FLOWS=')
    
    for schema_table in schemaS_tableS_lst:

        current_df = main_df[
            main_df['schemaS.tableS'] == schema_table
            ]
        
        print(current_df.head())

        schema_s = current_df.iloc[0]['SchemaS']
        source_table = current_df.iloc[0]['TableS']
        target_table = current_df.iloc[0]['TableT']
        
        query_full = ''
        query_prefix = 'select '
        query_suffix = ' from $schema.$table'
        query_cast_list = []

        for _, row in current_df.iterrows():
            target_column_name = row['CodeT']
            source_column_name = row['CodeS']
            source_column_type = row['Data Type']
            source_column_length = ''
            if row['Length'] and\
                row['Data Type'].lower() not in ('smallint',
                                                    'date',
                                                    'int',
                                                    'integer'):
                source_column_length = f"({row['Length']})"
            elif row['Data Type'].lower() == 'varchar2':
                source_column_length = '(255)'
            else:
                source_column_length = ''
            query_cast_list.append(
                f"cast('{source_column_name}' as "
                f"{source_column_type}{source_column_length}) as "
                f"'{target_column_name}'"
                )

        query_full = ', '.join(query_cast_list)

        query_full = query_prefix + query_full + query_suffix

        flow_template = {
            "loadType": "Scd1Replace",
            "source": {
                "schema": schema_s,
                "table": source_table,
                "query": query_full,
                "jdbcDialect": "OracleDialect"
            },
            "target": {
                "table": target_table
            }
        }
        
        if schtbl_cnt_trigger < schtbl_cnt_max:

            schtbl_cnt_trigger += 1

            test_flow_entity_lst.append(flow_template)
        
        else:
            
            schtbl_cnt_trigger = 0

            test_flow_entity_lst.append(flow_template)

            print_results(schema_t,
                                test_flow_entity_lst,
                                schtbl_num)
            
            schtbl_num += 1

            test_flow_entity_lst = []

elif db_type == 2:  # MSSQL
    # generate mssql flows
    print('=MAKING MSSQL FLOWS=')
    
    for schema_table in schemaS_tableS_lst:

        current_df = main_df[
            main_df['schemaS.tableS'] == schema_table
            ]

        schema_s = current_df.iloc[0]['SchemaS']
        table = current_df.iloc[0]['TableT']
        
        query_full = ''
        query_prefix = 'select '
        query_suffix = ' from $schema.$table'
        query_cast_list = []

        for _, row in current_df.iterrows():
            attr_f = row['CodeT']
            attr_l = row['CodeT']
            source_column_type = row['Data Type']
            source_column_length = ''
            if row['Length'] and\
                row['Data Type'].lower() not in ('smallint',
                                                'date',
                                                'int',
                                                'integer'):
                source_column_length = f"({row['Length']})"
            else:
                source_column_length = ''
            query_cast_list.append(
                f"cast('[{attr_f}]' as "
                f"{source_column_type}{source_column_length}) as "
                f"'[{attr_l}]'"
                )

        query_full = ', '.join(query_cast_list)

        query_full = query_prefix + query_full + query_suffix

        flow_template = {
            "loadType": "Scd1Replace",
            "source": {
                "schema": schema_s,
                "table": table,
                "query": query_full
            },
            "target": {
                "table": table
            }
        }

        if schtbl_cnt_trigger < schtbl_cnt_max:

            schtbl_cnt_trigger += 1

            test_flow_entity_lst.append(flow_template)
        
        else:
            
            schtbl_cnt_trigger = 0

            test_flow_entity_lst.append(flow_template)

            print_results(schema_t,
                                test_flow_entity_lst,
                                schtbl_num)
            
            schtbl_num += 1

            test_flow_entity_lst = []

# for last part of batch
if schtbl_cnt_trigger <= schtbl_len and schtbl_num > 1:
    print_results(schema_t,
                        test_flow_entity_lst,
                        schtbl_num)
# if mapping table count less than 200
if schtbl_cnt_trigger <= schtbl_len and schtbl_num == 1:
    schtbl_num = f'max_{schtbl_len}'
    print_results(schema_t,
                        test_flow_entity_lst,
                        schtbl_num)